# HW1. Задача 1: реакция пользователя

Нужно по диалогу определить реакцию пользователя: `negative`, `neutral` или `positive`. Ноутбук собран из того, что мы прошли на семинарах: Naive Bayes (занятие 1), логистическая регрессия и SVM (3), метрики, кросс-валидация, подбор гиперпараметров, тексты и TF-IDF, сохранение конвейера (4), SVD и ядра (5).

План:
1. данные и разведочный анализ
2. чистка и обработка текста
3. разбиение и метрики
4. baseline
5. три базовые модели и вложенная кросс-валидация
6. своя модель
7. своя гипотеза: какая часть диалога несёт реакцию
8. итоговая оценка на тестовом сете
9. разбор ошибок
10. сохранение моделей

## 0. Подготовка

In [2]:
import os, platform, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline

import solution_utils as su

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 110, "display.width", 200)
SEED = 42
DATA = os.environ.get("HW1_TRAIN", "./hw1_train.jsonl")
os.makedirs(su.ARTIFACTS, exist_ok=True)

In [3]:
# def show_table(frame, digits=3, axis=0):
#     """Таблица с заливкой: большие значения ярче.

#     axis=0 — каждый столбец со своей шкалой, axis=None — одна шкала на всю таблицу.
#     """
#     numeric = frame.select_dtypes("number").columns
#     styled = frame.style.background_gradient(cmap="Greens", subset=numeric, axis=axis).format(precision=digits, subset=numeric)
#     return styled

def show_table(frame, digits=3, axis=0):
    """Таблица без заливки (упрощенная версия)."""
    numeric = frame.select_dtypes("number").columns
    return frame.round(digits)

## 1. Данные и разведочный анализ

In [4]:
dialogs = su.read_dialogs(DATA)
frame_all = su.to_frame(dialogs)
print("диалогов:", len(frame_all), " размечено по реакции:", frame_all.reaction.notna().sum())
df = frame_all[frame_all.reaction.notna()].reset_index(drop=True)   # учим и проверяем только на размеченных
df[["dialog_id", "last_raw", "n_user", "reaction"]].head()

KeyboardInterrupt: 

In [4]:
display(df)

,dialog_id,text,all_user,last_user,first_user,all_asst,last_asst,last_asst_head,last_raw,n_user,len_last,len_asst,group,reaction
0,d094012177e,it s like but is undefined can you tell which modulus we can possibly use in this concept and into which i...,let s then try to mix it all together with some quantum physics terms in use this could result in some ext...,any spec symbols used in paradoxical arithmetic show or implement them all,it s like but is undefined can you tell which modulus we can possibly use in this concept and into which i...,introducing scientific terms from astrophysics or any other scientific discipline to the given equation ma...,paradoxical arithmetic typically involves the use of specialized symbols or notations that represent contr...,paradoxical arithmetic typically involves the use of specialized symbols or notations that represent contr...,"any spec symbols used in ""paradoxical arithmetic""? show or implement them all.",7,78,1498.857143,itslike112but11211isundefinedcanyoutellwhichmoduluswecanpossiblyuseinthisconcept,positive
1,d306fbe61fe,can you help me train a small language model that can answer questions i don t want to use a pretrained mo...,can you create me a python program to scrape article data from wikipedia and save them to a folder so i ca...,give me gui for this portion,can you help me train a small language model that can answer questions i don t want to use a pretrained mo...,creating a small language model from scratch is a significant undertaking as it involves understanding dee...,creating a gui for setting up and launching the training process of a neural network model involves buildi...,creating a gui for setting up and launching the training process of a neural network model involves buildi...,Give me GUI for this portion,8,28,4031.875000,canyouhelpmetrainasmalllanguagemodelthatcananswerquestionsidontwanttouseapretrai,neutral
2,dd6905952c7,ask questions about a fictional voice actor i made and based on the answers describe him how did this fict...,he was a loner in school and after getting out of college he accidentally bumpt into a director and gave h...,alex was at the supermarket when a fan bumped into him alex was really kind and humble alex signed a autog...,ask questions about a fictional voice actor i made and based on the answers describe him,how did this fictional voice actor get into the industry ? what is his most iconic role to date ? which ge...,based on the additional information provided here is a further description of the fictional voice actor al...,based on the additional information provided here is a further description of the fictional voice actor al...,"1.Alex was at the supermarket when a fan bumped into him,Alex was really kind and humble, Alex signed a au...",5,1484,2387.600000,askquestionsaboutafictionalvoiceactorimadeandbasedontheanswersdescribehim,neutral
3,d767a675896,calculate each prompt based off of data and show work calculate species richness shannon wiener index spec...,recalculate everything considering the porportions of each species and everything that gets affected by th...,recalculate everything considering the porportions of each species and everything that gets affected by th...,calculate each prompt based off of data and show work calculate species richness shannon wiener index spec...,to calculate species richness simply count the number of different species present in this case there are ...,apologies for the previous incorrect calculation let s recalculate everything considering the correct prop...,apologies for the previous incorrect calculation let s recalculate everything considering the correct prop...,"recalculate everything considering the porportions of each species, and everything that gets affected by t...",6,131,1542.166667,calculateeachpromptbasedoffofdataandshowworkcalculatespeciesrichnessshannonwiene,negative
4,d7a6280ce8e,mass spectrometer the length of ac rod will it affect to intensity ? longer or shorter better ? as a